# Grid, Field, and Operators

In this notebook, we explore the core data structures of the PDE framework:
- **Grid1D**: Uniform 1D computational domain
- **ScalarField**: Scalar field data on the grid
- **LaplacianOperator**: Discrete Laplacian operator
- **DirichletBC**: Dirichlet boundary conditions

We will demonstrate:
1. Creating a 1D grid
2. Initializing a scalar field with a Gaussian profile
3. Computing the discrete Laplacian
4. Applying Dirichlet boundary conditions

In [ ]:
# Setup: Import required modules
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

workspace_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pde_framework").exists():
        workspace_root = candidate
        break

if workspace_root is None:
    raise RuntimeError("Unable to locate the project root containing pde_framework")

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

import pde_framework as pde_framework_module

importlib.reload(pde_framework_module)

from pde_framework import (
    DirichletBC,
    DivergenceOperator,
    GradientOperator,
    Grid1D,
    LaplacianOperator,
    ScalarField,
    VectorField,
    gaussian_1d,
)

print(f"Setup complete. Public API imported successfully from {workspace_root}.")

## 1. Create 1D Grid

A uniform 1D grid spans from `x_min` to `x_max` with `n_points` discretization points.

In [ ]:
# Create the 1D grid
x_min, x_max, n_points = -1, 1, 201
grid = Grid1D(x_min=x_min, x_max=x_max, n_points=n_points)

print(f"Grid created: {grid.n_points} points from {grid.x_min} to {grid.x_max}")
print(f"Spacing (dx): {grid.dx:.6f}")
print(f"Grid shape: {grid.shape}")

# set visualization marker frequency
marker_frequency = 10

## 2. Initialize Scalar Field with Gaussian Profile

Create a Gaussian bell curve centered at x=0.

In [ ]:
# Initialize ScalarField with a Gaussian profile
field = ScalarField(grid)
field.apply_function(lambda x: gaussian_1d(x, center=0.0, sigma=0.2, amplitude=1.0))

print(f"Field initialized. Data shape: {field.data.shape}")
print(f"Field maximum: {field.data.max():.6f}")
print(f"Field norm: {field.norm():.6f}")

## 3. Plot Initial Field

Visualize the Gaussian profile.

In [ ]:
# Plot the scalar field
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    grid.x,
    field.data,
    color="tab:blue",
    linewidth=2,
    marker="o",
    markersize=3,
    markevery=marker_frequency,
    label="Gaussian profile",
)
ax.set_xlabel("x")
ax.set_ylabel("u(x)")
ax.set_title("Initial Scalar Field")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 4. Compute the Discrete Laplacian via Public API

Apply the public `LaplacianOperator` to the Gaussian field and visualize the result.

In [ ]:
# Compute and plot the discrete Laplacian via public API
laplacian_operator = LaplacianOperator()
laplacian_field = laplacian_operator.apply(field)

assert laplacian_field.data.shape == field.data.shape, (
    "Laplacian output shape must match field shape"
)
assert not np.allclose(laplacian_field.data[1:-1], 0.0), (
    "Gaussian Laplacian should not be identically zero"
)

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
axes[0].plot(
    grid.x,
    field.data,
    color="tab:blue",
    linewidth=2,
    marker="o",
    markersize=3,
    markevery=marker_frequency,
    label="Gaussian profile",
)
axes[0].set_ylabel("u(x)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    grid.x,
    laplacian_field.data,
    color="tab:orange",
    linewidth=2,
    marker="o",
    markersize=3,
    markevery=marker_frequency,
    label="Discrete Laplacian",
)
axes[1].set_xlabel("x")
axes[1].set_ylabel("u''(x)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle("Gaussian Field and Its Discrete Laplacian")
plt.tight_layout()
plt.show()

## 5. Apply Dirichlet Boundary Conditions via Public API

Apply fixed boundary values and visualize the effect on the field.

In [ ]:
# Apply Dirichlet boundary conditions using the public API
boundary_conditions = DirichletBC(left_value=0.0, right_value=0.0)
boundary_field = field.copy()
boundary_conditions.apply(boundary_field.data, boundary_field.grid)

assert np.isclose(boundary_field.data[0], 0.0), "Left boundary must be zero"
assert np.isclose(boundary_field.data[-1], 0.0), "Right boundary must be zero"

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    grid.x,
    field.data,
    color="tab:blue",
    linewidth=2,
    marker="o",
    markersize=3,
    markevery=5,
    label="Before BC",
)
ax.plot(
    grid.x,
    boundary_field.data,
    color="tab:red",
    linewidth=2,
    linestyle="--",
    marker="o",
    markersize=3,
    markevery=marker_frequency,
    label="After Dirichlet BC",
)
ax.set_xlabel("x")
ax.set_ylabel("u(x)")
ax.set_title("Effect of Dirichlet Boundary Conditions")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 6. Final Validation

Confirm the Laplacian and boundary-condition steps are consistent with the grid and public API.

In [ ]:
# Final notebook validation
assert field.data.shape == grid.shape, "Field data shape must match grid shape"
assert laplacian_field.data.shape == grid.shape, "Laplacian shape must match grid shape"
assert boundary_field.data.shape == grid.shape, "Boundary field shape must match grid shape"
assert np.isclose(boundary_field.data[0], 0.0)
assert np.isclose(boundary_field.data[-1], 0.0)

print("Final notebook validation passed.")

## 7. Gradient and Divergence (Epic 8)

Now we extend the notebook with first-order differential operators:

- `GradientOperator` maps a `ScalarField` to a `VectorField`.
- `DivergenceOperator` maps a `VectorField` to a `ScalarField`.

We demonstrate:
1. Gradient of the Gaussian profile.
2. Divergence of a simple 1D vector field `v(x)=x`.

In [ ]:
# Compute gradient of the Gaussian scalar field
gradient_operator = GradientOperator(scheme="central")
gradient_field = gradient_operator.apply(field)

assert isinstance(gradient_field, VectorField)
assert len(gradient_field.components) == 1
assert gradient_field.components[0].shape == grid.shape

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(grid.x, gradient_field.components[0], color="tab:green", linewidth=2, label="grad(u)")
ax.set_title("Gradient of Gaussian profile")
ax.set_xlabel("x")
ax.set_ylabel("du/dx")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

# Divergence example on v(x)=x
vector_linear = VectorField(grid, [grid.x.copy()])
divergence_operator = DivergenceOperator()
divergence_field = divergence_operator.apply(vector_linear)

assert divergence_field.data.shape == grid.shape
assert np.allclose(divergence_field.data[1:-1], 1.0, atol=1e-2)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(grid.x, vector_linear.components[0], color="tab:purple", linewidth=2, label="v(x)=x")
ax.plot(grid.x, divergence_field.data, color="tab:red", linewidth=2, linestyle="--", label="div(v)")
ax.set_title("Divergence of a simple 1D vector field")
ax.set_xlabel("x")
ax.set_ylabel("value")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

print("Epic 8 notebook checks passed.")